# Lesson 3.3 — Disaster Recovery Patterns (RTO/RPO)

**Level: Intermediate** | **Exam:** SOA-C03 Domain 2 Task 2.3; SAP-C02 Domain 2 Task 2.2

## Learning Objectives
- Define RTO and RPO and use them to drive DR architecture decisions
- Implement and compare the four AWS DR patterns: Backup/Restore, Pilot Light, Warm Standby, Active/Active
- Calculate cost vs. recovery time trade-offs for each DR strategy
- Configure AWS Elastic Disaster Recovery (DRS) for workload DR

_Reference: [AWS DR Whitepaper](https://docs.aws.amazon.com/whitepapers/latest/disaster-recovery-workloads-on-aws/) | [AWS Elastic DR](https://docs.aws.amazon.com/drs/latest/userguide/)_

## RTO and RPO

- **RTO (Recovery Time Objective)**: Maximum acceptable **downtime** — how long can the business tolerate the service being down?
- **RPO (Recovery Point Objective)**: Maximum acceptable **data loss** — how much data can we afford to lose?

```
Timeline:
──────────────────────────────────────────────────────────────
   Disaster       Recovery starts     Service restored
   event │              │                   │
─────────┤──────────────┤───────────────────┤─────────────────
         │← ─ ─ ─ RTO ─ ─ ─ ─ ─ ─ ─ ─ ─ →│
Last backup                                │
   │← RPO →│ Disaster event                │
```

## The Four DR Patterns

| Strategy | RTO | RPO | Cost | How it works |
|----------|-----|-----|------|--------------|
| **Backup & Restore** | Hours | Hours | $ | Periodic backups to S3/Glacier; restore from scratch |
| **Pilot Light** | 10s of min | Minutes | $$ | Core DB always running; scale app servers on disaster |
| **Warm Standby** | Minutes | Seconds | $$$ | Scaled-down copy always running; scale up on disaster |
| **Multi-Site Active/Active** | Near-zero | Near-zero | $$$$ | Full capacity in multiple regions, always serving |

### Pattern Details

**Backup & Restore**: Cheapest, highest RTO. Use when data can be reconstructed and downtime of hours is acceptable.
```
Normal: → Primary Region (production only)
Nightly backups → S3 (cross-region replication)
Disaster: Restore from S3 to DR region (hours)
```

**Pilot Light**: Database continuously replicated; app servers are stopped.
```
Normal: → Primary Region
         RDS replicated to DR region (continuous)
         EC2/ECS in DR region = STOPPED
Disaster: Start/scale EC2, point DNS to DR region (tens of minutes)
```

**Warm Standby**: Scaled-down version always running at minimum capacity.
```
Normal: → Primary Region (full scale)
         DR region at 1/4 capacity, continuously synced
Disaster: Scale up DR region, switchover DNS (minutes)
```

**Active/Active**: Highest cost, near-zero RTO/RPO. All regions serve traffic simultaneously.
```
Normal: → Traffic split across regions via Route 53
         DynamoDB Global Tables / Aurora Global Database
Disaster: Route 53 removes failed region from routing (seconds)
```

In [ ]:
import boto3, json

backup = boto3.client('backup', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# DR Pattern decision matrix
import json

dr_patterns = [
    {
        'pattern': 'Backup & Restore',
        'rto': 'Hours (2-24h)',
        'rpo': 'Hours (last backup)',
        'cost_multiplier': '1x',
        'aws_services': ['AWS Backup', 'S3 Cross-Region Replication', 'CloudFormation'],
        'use_case': 'Non-critical apps, development, data archives'
    },
    {
        'pattern': 'Pilot Light',
        'rto': '10s of minutes (30-60 min)',
        'rpo': 'Minutes (near real-time DB sync)',
        'cost_multiplier': '2x (DB always running in DR)',
        'aws_services': ['RDS Read Replica → promote', 'CloudFormation', 'Route 53 DNS failover'],
        'use_case': 'Internal business apps, RPO of minutes acceptable'
    },
    {
        'pattern': 'Warm Standby',
        'rto': 'Minutes (5-15 min scale up)',
        'rpo': 'Seconds to minutes',
        'cost_multiplier': '3-4x (reduced capacity in DR)',
        'aws_services': ['ASG with min capacity', 'Aurora Global Database', 'Route 53 health checks'],
        'use_case': 'Customer-facing apps, e-commerce, financial services'
    },
    {
        'pattern': 'Multi-Site Active/Active',
        'rto': 'Seconds (DNS TTL only)',
        'rpo': 'Near-zero (sub-second with Global Tables)',
        'cost_multiplier': '4x+ (full capacity everywhere)',
        'aws_services': ['DynamoDB Global Tables', 'Aurora Global DB', 'Route 53 latency routing'],
        'use_case': 'Mission-critical: banking, healthcare, global platforms'
    }
]

print('Disaster Recovery Pattern Comparison:\n')
for p in dr_patterns:
    print(f"{'='*60}")
    print(f"Pattern: {p['pattern']}")
    print(f"  RTO: {p['rto']}")
    print(f"  RPO: {p['rpo']}")
    print(f"  Cost: {p['cost_multiplier']}")
    print(f"  Services: {', '.join(p['aws_services'])}")
    print(f"  Use: {p['use_case']}")

In [ ]:
# AWS Backup — create a backup vault and plan

VAULT_NAME = 'cloudops-learning-vault'

try:
    vault = backup.create_backup_vault(BackupVaultName=VAULT_NAME)
    print(f'Created backup vault: {VAULT_NAME}')
except backup.exceptions.AlreadyExistsException:
    print(f'Vault already exists: {VAULT_NAME}')

# Create a backup plan with cross-region copy
backup_plan = {
    'BackupPlanName': 'cloudops-learning-plan',
    'Rules': [
        {
            'RuleName': 'DailyBackupWithWeeklyRetention',
            'TargetBackupVaultName': VAULT_NAME,
            'ScheduleExpression': 'cron(0 5 ? * * *)',   # Daily at 5 AM UTC
            'StartWindowMinutes': 60,
            'CompletionWindowMinutes': 120,
            'Lifecycle': {
                'DeleteAfterDays': 35   # Retain daily backups for 35 days
            },
            'CopyActions': [
                {
                    'DestinationBackupVaultArn': f'arn:aws:backup:us-west-2:{boto3.client("sts").get_caller_identity()["Account"]}:backup-vault:Default',
                    'Lifecycle': {'DeleteAfterDays': 90}   # Keep cross-region copy 90 days
                }
            ]
        }
    ]
}

try:
    plan = backup.create_backup_plan(BackupPlan=backup_plan)
    print(f'Created backup plan: {plan["BackupPlanId"]}')
except backup.exceptions.AlreadyExistsException:
    print('Backup plan already exists.')
except Exception as e:
    print(f'Backup plan: {e}')

## Exercise

In [ ]:
# TODO: A customer gives you these requirements. Recommend the DR pattern and justify your choice.
# Fill in the recommendation dict for each scenario.

requirements = [
    {
        'customer': 'Online banking platform',
        'rto_requirement': '< 1 minute',
        'rpo_requirement': '< 1 second',
        'budget': 'High — mission critical',
        'recommendation': 'TODO',  # Which DR pattern?
        'aws_services': 'TODO',    # Which services?
        'justification': 'TODO'
    },
    {
        'customer': 'Internal HR portal',
        'rto_requirement': '< 4 hours',
        'rpo_requirement': '< 1 hour',
        'budget': 'Low — cost-sensitive',
        'recommendation': 'TODO',
        'aws_services': 'TODO',
        'justification': 'TODO'
    },
]

for r in requirements:
    print(f"Customer: {r['customer']}")
    print(f"  RTO: {r['rto_requirement']} | RPO: {r['rpo_requirement']} | Budget: {r['budget']}")
    print(f"  Recommended DR Pattern: {r['recommendation']}")
    print(f"  AWS Services: {r['aws_services']}")
    print(f"  Justification: {r['justification']}")
    print()

In [ ]:
# Cleanup
import boto3
backup = boto3.client('backup', region_name='us-east-1')

try:
    plans = backup.list_backup_plans()['BackupPlansList']
    for p in plans:
        if p['BackupPlanName'] == 'cloudops-learning-plan':
            backup.delete_backup_plan(BackupPlanId=p['BackupPlanId'])
            print('Deleted backup plan')
except Exception as e:
    print(f'Plan cleanup: {e}')

try:
    backup.delete_backup_vault(BackupVaultName='cloudops-learning-vault')
    print('Deleted backup vault')
except Exception as e:
    print(f'Vault cleanup: {e}')

## Summary
- **RTO** = max acceptable downtime; **RPO** = max acceptable data loss
- Four DR patterns: Backup/Restore ($ hours) → Pilot Light ($$ tens of min) → Warm Standby ($$$ min) → Active/Active ($$$$ seconds)
- **AWS Backup** centralizes backup policies across EC2, RDS, DynamoDB, EFS, and S3
- For multi-region active/active: **DynamoDB Global Tables** (NoSQL) or **Aurora Global Database** (relational)

---
**Next:** [Lesson 3.4 — AWS Backup, Snapshots & Data Versioning](lesson-3.4-backup-restore.ipynb)